# 🧠 RAG Förderkatalog v0.2.0 - Google Colab

[![GitHub Release](https://img.shields.io/badge/release-v0.2.0-blue.svg)](https://github.com/dgaida/rag_foerderkatalog/releases/tag/v0.2.0)
[![Python 3.11+](https://img.shields.io/badge/python-3.11+-blue.svg)](https://www.python.org/downloads/)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

**Semantische Suche in deutschen Forschungsförderprojekten**

Dieses Notebook ermöglicht die einfache Nutzung der RAG Förderkatalog-Anwendung in Google Colab:
- ✅ **HuggingFace Embeddings** statt Ollama (Cloud-kompatibel)
- ✅ **Vorbereiteter Index** (~300k Projekte)
- ✅ **Keine lokale Installation** nötig
- ✅ **Gradio Web-UI** im Browser

---

## 📋 Voraussetzungen

- **GROQ API Key** für LLM-Funktionen (kostenlos unter [console.groq.com](https://console.groq.com/))
- **Google Drive** wird temporär für Downloads genutzt (~2GB)
- **Runtime**: Standard-Python (kein GPU nötig)

---

## 🚀 Schritt 1: Installation der Dependencies

In [ ]:
%%capture
# Basis-Pakete installieren (dauert ~2-3 Minuten)
!pip install --upgrade pip setuptools wheel

# Core Dependencies
!pip install pandas numpy faiss-cpu gradio python-dotenv tqdm requests

# LLM Client
!pip install git+https://github.com/dgaida/llm_client.git

# HuggingFace Embeddings Support
!pip install llama-index-embeddings-huggingface

print("✅ Dependencies installiert!")

In [ ]:
%%capture
# RAG Förderkatalog v0.2.0 installieren
!pip install git+https://github.com/dgaida/rag_foerderkatalog.git@v0.2.0

print("✅ RAG Förderkatalog v0.2.0 installiert!")

## 📥 Schritt 2: Download des vorbereiteten Index

In [ ]:
import os
import zipfile
from pathlib import Path
import requests
from tqdm import tqdm

# Erstelle Verzeichnisse
os.makedirs('data', exist_ok=True)
os.makedirs('input', exist_ok=True)

# Download URL
INDEX_URL = "https://github.com/dgaida/rag_foerderkatalog/releases/download/v0.2.0/rag_foerderkatalog_index_v0.2.0.zip"
ZIP_FILE = "rag_index_v0.2.0.zip"

print("📥 Lade vorbereiteten Index herunter...")
print(f"   URL: {INDEX_URL}")
print(f"   Größe: ~800 MB")
print("")

# Download mit Fortschrittsanzeige
response = requests.get(INDEX_URL, stream=True)
total_size = int(response.headers.get('content-length', 0))

with open(ZIP_FILE, 'wb') as f, tqdm(
    desc="Download",
    total=total_size,
    unit='B',
    unit_scale=True,
    unit_divisor=1024,
) as pbar:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)
        pbar.update(len(chunk))

print("")
print("📦 Entpacke Index...")

# Entpacken
with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
    zip_ref.extractall('data/')

# Aufräumen
os.remove(ZIP_FILE)

# Prüfe ob Dateien existieren
index_file = Path('data/vector_hf.index')
map_file = Path('data/embeddings_map_hf.json')

if index_file.exists() and map_file.exists():
    print("")
    print("✅ Index erfolgreich geladen!")
    print(f"   • Index: {index_file} ({index_file.stat().st_size / 1e9:.2f} GB)")
    print(f"   • Mapping: {map_file} ({map_file.stat().st_size / 1e6:.2f} MB)")
else:
    print("❌ Fehler: Index-Dateien nicht gefunden!")
    raise FileNotFoundError("Index-Dateien fehlen nach dem Entpacken")

## 📄 Schritt 3: CSV-Datei herunterladen

Die CSV-Datei wird vom BMBF Förderkatalog heruntergeladen (~200 MB).

In [ ]:
import requests
from tqdm import tqdm

# BMBF Förderkatalog URL
CSV_URL = "https://foerderportal.bund.de/foekat/export/foerderkatalog_export.csv"
CSV_FILE = "input/foerderkatalog_export.csv"

print("📥 Lade BMBF Förderkatalog CSV...")
print(f"   URL: {CSV_URL}")
print("")

# Download mit Fortschrittsanzeige
response = requests.get(CSV_URL, stream=True)
total_size = int(response.headers.get('content-length', 0))

with open(CSV_FILE, 'wb') as f, tqdm(
    desc="Download CSV",
    total=total_size,
    unit='B',
    unit_scale=True,
    unit_divisor=1024,
) as pbar:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)
        pbar.update(len(chunk))

csv_path = Path(CSV_FILE)
if csv_path.exists():
    print("")
    print(f"✅ CSV erfolgreich heruntergeladen!")
    print(f"   • Datei: {csv_path}")
    print(f"   • Größe: {csv_path.stat().st_size / 1e6:.2f} MB")
else:
    print("❌ Fehler: CSV konnte nicht heruntergeladen werden!")
    raise FileNotFoundError("CSV-Datei fehlt")

## 🔑 Schritt 4: API Key konfigurieren

Erstellen Sie einen kostenlosen GROQ API Key unter: [console.groq.com](https://console.groq.com/)

In [ ]:
import os
from getpass import getpass

# API Key eingeben (wird nicht angezeigt)
print("🔑 GROQ API Key konfigurieren")
print("")
print("Erstellen Sie einen kostenlosen Account unter:")
print("👉 https://console.groq.com/")
print("")

api_key = getpass("Geben Sie Ihren GROQ API Key ein: ")

# Setze Umgebungsvariable
os.environ['GROQ_API_KEY'] = api_key

# Erstelle .env Datei
with open('.env', 'w') as f:
    f.write(f'GROQ_API_KEY={api_key}\n')

print("")
print("✅ API Key konfiguriert!")
print("")
print("⚠️  Hinweis: Der Key wird nur für diese Session gespeichert.")

## 🚀 Schritt 5: Anwendung starten

Die Gradio-Oberfläche wird automatisch geöffnet. Klicken Sie auf den generierten Link.

In [ ]:
from src.search.engine import ProjectSearchEngine
from src.app import build_ui
from src.utils.logging_config import setup_logging
import logging

# Setup Logging
setup_logging(level=logging.INFO)

print("🧠 RAG Förderkatalog v0.2.0")
print("="*60)
print("")
print("🔧 Konfiguration:")
print("   • Provider: HuggingFace")
print("   • Modell: intfloat/e5-small-v2")
print("   • Index: Pre-loaded (v0.2.0)")
print("")
print("⏳ Initialisiere Engine...")

# Engine mit HuggingFace Provider initialisieren
engine = ProjectSearchEngine(
    provider="huggingface"
)

# CSV laden
print("📊 Lade CSV-Daten...")
engine.load_and_clean()

# Index-Info anzeigen
info = engine.get_index_info()
print("")
print("📈 Index-Informationen:")
print(f"   • CSV-Zeilen: {info['csv_rows']:,}")
print(f"   • Indizierte Vektoren: {info['total_vectors']:,}")
print(f"   • Embedding-Dimension: {info['dimension']}")
print(f"   • Abdeckung: {(info['total_vectors']/info['csv_rows']*100):.1f}%")
print("")
print("🌐 Starte Gradio-Oberfläche...")
print("")
print("👉 Klicken Sie auf den generierten Link unten!")
print("")

# Gradio UI starten
demo = build_ui(engine)
demo.launch(
    share=True,  # Öffentlicher Link (Colab-kompatibel)
    debug=False,
    show_error=True
)

## 💡 Nutzungshinweise

### Suchmodi

- **Hybrid** (empfohlen): Kombiniert semantische und Keyword-Suche
- **Semantic**: Reine KI-basierte Vektorsuche
- **Keyword**: Schnelle textbasierte Suche

### Beispielsuchen

```
Künstliche Intelligenz Hochschule Bayern
Wasserstoff Energie NRW 2020-2025
Quantencomputing Forschung
Klimawandel Digitalisierung
Medizintechnik Berlin
```

### Features

- ✅ **300.000+ Förderprojekte** durchsuchbar
- ✅ **Semantische Suche** findet thematisch ähnliche Projekte
- ✅ **KI-Analyse** generiert Zusammenfassungen
- ✅ **Projekt-Details** per FKZ-Auswahl
- ✅ **Statistiken** zu Fördersummen und Zeiträumen

---

## 🛠️ Fehlerbehebung

### Problem: "Out of Memory"

**Lösung**: Starten Sie die Runtime neu und führen Sie nur die nötigen Zellen aus.

```python
# Runtime neu starten
from IPython import get_ipython
get_ipython().kernel.do_shutdown(True)
```

### Problem: "API Key ungültig"

**Lösung**: Überprüfen Sie Ihren GROQ API Key:

```python
import os
print(f"API Key: {os.environ.get('GROQ_API_KEY', 'NICHT GESETZT')}")
```

### Problem: "Index nicht gefunden"

**Lösung**: Führen Sie Schritt 2 (Download) erneut aus.

```python
# Prüfe Index-Dateien
from pathlib import Path
print(f"Index existiert: {Path('data/vector_hf.index').exists()}")
print(f"Mapping existiert: {Path('data/embeddings_map_hf.json').exists()}")
```

### Problem: "HuggingFace Model Download langsam"

**Lösung**: Das erste Laden des Modells dauert 1-2 Minuten. Das ist normal.

---

## ℹ️ Weitere Informationen

### Links

- 📦 [GitHub Repository](https://github.com/dgaida/rag_foerderkatalog)
- 📖 [Dokumentation](https://github.com/dgaida/rag_foerderkatalog#readme)
- 🐛 [Issues](https://github.com/dgaida/rag_foerderkatalog/issues)
- 💬 [Discussions](https://github.com/dgaida/rag_foerderkatalog/discussions)

### Technische Details

- **Python**: 3.11+
- **Embeddings**: HuggingFace (intfloat/e5-small-v2, 384 dim)
- **Vector DB**: FAISS (CPU)
- **LLM**: GROQ API
- **UI**: Gradio 4.0+

### Lizenz

MIT License - siehe [LICENSE](https://github.com/dgaida/rag_foerderkatalog/blob/master/LICENSE)

---

**© 2025 RAG Förderkatalog** | v0.2.0
